# 🛒 Walmart Retail Sales — Data Preprocessing & EDA Pipeline

---

## 📌 Project Overview

This notebook implements a **full end-to-end data preprocessing and exploratory data analysis (EDA) pipeline** for the Walmart retail sales dataset.

### 📂 Input Files
| File | Description |
|------|-------------|
| `features.xlsx` | Economic indicators and MarkDown promotions per store/date |
| `stores.csv` | Store metadata (Type, Size) |
| `train.csv` | Historical weekly sales per department/store |
| `test.csv` | Future dates for prediction |

### 🔄 Pipeline Order
```
1. Load Datasets
2. Data Cleaning
3. Feature Engineering (MarkDown)
4. Normalization (MinMaxScaler)
5. Encoding Categorical/Boolean Features
6. Merging Datasets → full_df
7. EDA & Visualizations
```

> ⚠️ **Run all cells top-to-bottom.** Each section depends on the previous one.

---

## 📦 Section 1 — Import Libraries

### 🎯 Objective
Import all required libraries upfront so that no cell fails due to missing imports.

### 🧠 Why?
Centralizing imports makes the notebook easier to maintain and avoids redundant `import` statements scattered across cells.

In [1]:
# Core data manipulation
import pandas as pd
import numpy as np

# Machine learning preprocessing
from sklearn.preprocessing import MinMaxScaler

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')

print('✅ All libraries imported successfully.')

✅ All libraries imported successfully.


---

## 📂 Section 2 — Load Datasets

### 🎯 Objective
Load all four source files into Pandas DataFrames.

### 🧠 Why?
- `features.xlsx` is in Excel format — we use `read_excel()`.
- The three CSV files use `read_csv()`.
- We inspect each DataFrame's shape and first rows to confirm successful loading.

In [2]:
# ── Load features from Excel ──────────────────────────────────────────────────
features_df = pd.read_excel('features.xlsx')
print(f'features_df  → shape: {features_df.shape}')
print(f'  columns: {features_df.columns.tolist()}\n')

# ── Load CSV files ────────────────────────────────────────────────────────────
train_df  = pd.read_csv('train.csv')
stores_df = pd.read_csv('stores.csv')
test_df   = pd.read_csv('test.csv')

print(f'train_df   → shape: {train_df.shape}')
print(f'stores_df  → shape: {stores_df.shape}')
print(f'test_df    → shape: {test_df.shape}')

features_df  → shape: (8190, 12)
  columns: ['Store', 'Date', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'IsHoliday']

train_df   → shape: (421570, 5)
stores_df  → shape: (45, 3)
test_df    → shape: (115064, 4)


In [3]:
features_df.head()

,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


In [4]:
train_df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


In [5]:
stores_df.head()

,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


---

## 🧹 Section 3 — Data Cleaning

### 🎯 Objective
Parse date columns into proper `datetime` dtype and verify there are no unexpected missing values in key join columns.

### 🧠 Why?
- Date columns loaded from CSV/Excel are strings by default. Merging on string dates can silently fail if formats differ between DataFrames.
- Detecting NaN in join keys (`Store`, `Date`, `IsHoliday`) early prevents silent data loss during the merge step.

### ⚙️ What the code does
1. Converts `Date` in `features_df`, `train_df`, and `test_df` to `datetime64`.
2. Prints a quick NaN summary for each DataFrame.

In [6]:
# ── Parse Date columns ────────────────────────────────────────────────────────
features_df['Date'] = pd.to_datetime(features_df['Date'])
train_df['Date']    = pd.to_datetime(train_df['Date'])

if 'Date' in test_df.columns:
    test_df['Date'] = pd.to_datetime(test_df['Date'])
else:
    print("⚠️ test_df has no 'Date' column. Available:", test_df.columns.tolist())

print('✅ Date columns parsed.')

# ── Null check ────────────────────────────────────────────────────────────────
for name, df in [('features_df', features_df), ('train_df', train_df),
                 ('stores_df', stores_df)]:
    n_null = df.isnull().sum().sum()
    print(f'{name}: {n_null} total NaN values')

✅ Date columns parsed.
features_df: 24040 total NaN values
train_df: 0 total NaN values
stores_df: 0 total NaN values


In [7]:
# ── Date coverage check ───────────────────────────────────────────────────────
features_dates = set(features_df['Date'].unique())
train_dates    = set(train_df['Date'].unique())
common_dates   = features_dates & train_dates

print(f'Unique dates — features_df : {len(features_dates)}')
print(f'Unique dates — train_df    : {len(train_dates)}')
print(f'Common dates               : {len(common_dates)}')

only_features = features_dates - train_dates
only_train    = train_dates - features_dates

if only_features:
    print(f'\nDates only in features_df ({len(only_features)}): {sorted(only_features)[:5]} ...')
if only_train:
    print(f'Dates only in train_df ({len(only_train)}): {sorted(only_train)[:5]} ...')

Unique dates — features_df : 182
Unique dates — train_df    : 143
Common dates               : 143

Dates only in features_df (39): [Timestamp('2012-11-02 00:00:00'), Timestamp('2012-11-09 00:00:00'), Timestamp('2012-11-16 00:00:00'), Timestamp('2012-11-23 00:00:00'), Timestamp('2012-11-30 00:00:00')] ...


---

## 🔧 Section 4 — Feature Engineering: MarkDown

### 🎯 Objective
Consolidate the five individual MarkDown columns (`MarkDown1`–`MarkDown5`) into a single `MarkDown` feature.

### 🧠 Why?
- **Dimensionality reduction** — Five sparse columns become one dense, informative feature.
- **Capturing total promotional intensity** — The sum represents the total discount/promotion applied across all markdown types.
- **Stability** — Combining columns reduces the noise individual markdowns introduce.

### ⚙️ What the code does
1. Fills NaN → `0` (no promotion recorded = no promotion).
2. Clips negative values → `0` (negative promotions are noise, not real discounts).
3. Sums all five columns → new `MarkDown` column.
4. Drops the original five columns to keep the dataset clean.

> ✅ **This step is performed on `features_df` BEFORE normalization and merging**, ensuring `MarkDown` propagates correctly into the final dataset.

In [8]:
markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

# Confirm these columns exist before proceeding
missing = [c for c in markdown_cols if c not in features_df.columns]
if missing:
    print(f'⚠️ Missing MarkDown columns: {missing}')
else:
    # Step 1 — Fill NA with 0 (no markdown = no promotion)
    features_df[markdown_cols] = features_df[markdown_cols].fillna(0)

    # Step 2 — Clip negatives to 0 (treat as noise)
    features_df[markdown_cols] = features_df[markdown_cols].clip(lower=0)

    # Step 3 — Combine into single MarkDown feature
    features_df['MarkDown'] = features_df[markdown_cols].sum(axis=1)

    # Step 4 — Drop original columns
    features_df.drop(columns=markdown_cols, inplace=True)

    print('✅ MarkDown feature engineered successfully.')
    print(f'   MarkDown range: [{features_df["MarkDown"].min():.2f}, {features_df["MarkDown"].max():.2f}]')
    print(f'   features_df columns now: {features_df.columns.tolist()}')

✅ MarkDown feature engineered successfully.
   MarkDown range: [0.00, 783529.45]
   features_df columns now: ['Store', 'Date', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'IsHoliday', 'MarkDown']


---

## ⚖️ Section 5 — Normalization (MinMaxScaler)

### 🎯 Objective
Scale all continuous numerical features in `features_df` to the [0, 1] range.

### 🧠 Why?
- Many ML algorithms (KNN, SVM, gradient-based models) are sensitive to feature scale.
- Min-Max scaling preserves the original distribution shape while bounding values to [0, 1].
- We normalize **once** here, **before** merging, to avoid accidentally scaling the same column twice.

### ⚙️ What the code does
Applies `MinMaxScaler` independently to each of: `Temperature`, `Fuel_Price`, `CPI`, `Unemployment`, and `MarkDown`.

> ⚠️ **We do NOT apply normalization again after merging.** This is a common source of double-scaling bugs — fixed here.

In [9]:
columns_to_normalize = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'MarkDown']

# Verify all columns exist in features_df
available = [c for c in columns_to_normalize if c in features_df.columns]
skipped   = [c for c in columns_to_normalize if c not in features_df.columns]

if skipped:
    print(f'⚠️ Columns not found (skipped): {skipped}')

scaler = MinMaxScaler()

# Apply a single scaler — fit_transform each column independently
for col in available:
    features_df[col] = scaler.fit_transform(features_df[[col]])

print(f'✅ Normalized columns: {available}')
features_df[available].describe().round(4)

✅ Normalized columns: ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'MarkDown']


,Temperature,Fuel_Price,CPI,Unemployment,MarkDown
count,8190.0000,8190.0000,7605.0000,7605.0000,8190.0000
mean,0.6101,0.4679,0.4508,0.3898,0.0113
std,0.1710,0.2161,0.3861,0.1766,0.0244
min,0.0000,0.0000,0.0000,0.0000,0.0000
25%,0.4869,0.2851,0.0612,0.2775,0.0000
50%,0.6225,0.5215,0.5510,0.3878,0.0000
75%,0.7430,0.6368,0.8538,0.4594,0.0151
max,1.0000,1.0000,1.0000,1.0000,1.0000


---

## 🔢 Section 6 — Encoding Boolean Features

### 🎯 Objective
Convert the `IsHoliday` boolean column to integer (0/1) in both `features_df` and `train_df`.

### 🧠 Why?
Machine Learning models require numeric input. Boolean `True/False` values can cause type errors or unexpected behavior in many scikit-learn estimators.

### ⚙️ What the code does
Casts `IsHoliday` to `int` in both DataFrames. This must happen **before** the merge so the join key types match.

In [10]:
# Encode IsHoliday in features_df
if 'IsHoliday' in features_df.columns:
    features_df['IsHoliday'] = features_df['IsHoliday'].astype(int)

# Encode IsHoliday in train_df
if 'IsHoliday' in train_df.columns:
    train_df['IsHoliday'] = train_df['IsHoliday'].astype(int)

print('✅ IsHoliday encoded to int in features_df and train_df.')
print(f'   features_df IsHoliday dtype: {features_df["IsHoliday"].dtype}')
print(f'   train_df    IsHoliday dtype: {train_df["IsHoliday"].dtype}')

✅ IsHoliday encoded to int in features_df and train_df.
   features_df IsHoliday dtype: int64
   train_df    IsHoliday dtype: int64


---

## 🔗 Section 7 — Merging Datasets → `full_df`

### 🎯 Objective
Build the unified `full_df` DataFrame that contains:
- Weekly sales (`Weekly_Sales`)
- All economic features (`Temperature`, `Fuel_Price`, `CPI`, `Unemployment`)
- Promotional feature (`MarkDown`)
- Store metadata (`Type`, `Size`)
- Holiday indicator (`IsHoliday`)

### 🧠 Why?
ML models train on a single flat table. The source data is normalized across three relational files that must be joined correctly.

### ⚙️ Merge Strategy
```
Step 1: features_df  ← LEFT JOIN stores_df   ON [Store]
Step 2: features_merged ← LEFT JOIN train_df  ON [Store, Date, IsHoliday]  (reversed direction)
         → i.e., train_df LEFT JOIN features_merged ON [Store, Date, IsHoliday]
```

> 📌 We start the final merge from `train_df` (the sales table) to ensure every sales record is kept, and features are pulled in as available.

In [ ]:
# ── Step 1: Merge store metadata into features ────────────────────────────────
features_merged = features_df.merge(
    stores_df,
    on='Store',
    how='left'
)

print(f'features_merged shape: {features_merged.shape}')
print(f'columns: {features_merged.columns.tolist()}')

In [ ]:
# ── Step 2: Merge features into train_df to create full_df ────────────────────
# Starting from train_df ensures every Weekly_Sales row is preserved.
full_df = train_df.merge(
    features_merged,
    on=['Store', 'Date', 'IsHoliday'],
    how='left'
)

print(f'\nfull_df shape    : {full_df.shape}')
print(f'full_df columns  : {full_df.columns.tolist()}')

# ── Confirm MarkDown exists ───────────────────────────────────────────────────
if 'MarkDown' in full_df.columns:
    print(f'\n✅ MarkDown present in full_df.')
    print(f'   NaN count: {full_df["MarkDown"].isna().sum()}')
else:
    print('❌ MarkDown NOT found in full_df — check merge keys.')

In [ ]:
# ── Post-merge NaN audit ──────────────────────────────────────────────────────
null_summary = full_df.isnull().sum()
null_summary = null_summary[null_summary > 0]

if null_summary.empty:
    print('✅ No NaN values in full_df.')
else:
    print('⚠️ NaN values detected after merge:')
    print(null_summary)

full_df.head()

---

## 🚨 Section 8 — Outlier Detection

### 🎯 Objective
Visualize the distribution and outliers of the four economic features using **boxplots** and **histograms**.

### 🧠 Why?
- Outliers in features like `CPI` or `Unemployment` may distort model training.
- We use the IQR method to *detect* outliers visually before deciding whether to remove them.

### ⚠️ Important
We only inspect statistical outliers in numeric features, while preserving meaningful business variations (holidays, promotions).

> 📊 The data is already normalized to [0, 1] at this stage, so the axes will reflect normalized values.

In [ ]:
num_cols = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Boxplots of Normalized Economic Features', fontsize=14, fontweight='bold')

for i, col in enumerate(num_cols):
    ax = axes[i // 2, i % 2]
    sns.boxplot(y=features_df[col], ax=ax)
    ax.set_title(f'Boxplot of {col}')
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Histograms of Normalized Economic Features', fontsize=14, fontweight='bold')

for i, col in enumerate(num_cols):
    ax = axes[i // 2, i % 2]
    sns.histplot(features_df[col], bins=30, kde=True, ax=ax)
    ax.set_title(f'Histogram of {col}')
    ax.set_xlabel(col)

plt.tight_layout()
plt.show()

---

## 📊 Section 9 — EDA: Weekly Sales vs Holiday

### 🎯 Objective
Understand how `IsHoliday` (holiday vs non-holiday weeks) impacts `Weekly_Sales`.

### 🧠 Why?
Holiday weeks are expected to have higher sales due to increased consumer spending. Confirming this pattern validates our feature and helps the model weigh it appropriately.

### ⚙️ What the code does
1. Boxplot — distribution of weekly sales split by holiday flag.
2. Bar chart — mean weekly sales per group.

In [ ]:
# Ensure correct type
full_df['IsHoliday'] = full_df['IsHoliday'].astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Weekly Sales: Holiday vs Non-Holiday', fontsize=14, fontweight='bold')

# ── Boxplot ───────────────────────────────────────────────────────────────────
sns.boxplot(
    x='IsHoliday', y='Weekly_Sales', data=full_df,
    ax=axes[0]
)
axes[0].set_title('Distribution of Weekly Sales')
axes[0].set_xlabel('Is Holiday (0 = No, 1 = Yes)')
axes[0].set_ylabel('Weekly Sales')

# ── Bar chart (mean) ─────────────────────────────────────────────────────────
avg_sales = full_df.groupby('IsHoliday')['Weekly_Sales'].mean()
sns.barplot(
    x=avg_sales.index, y=avg_sales.values,
    ax=axes[1]
)
axes[1].set_title('Average Weekly Sales')
axes[1].set_xlabel('Is Holiday (0 = No, 1 = Yes)')
axes[1].set_ylabel('Average Weekly Sales')

plt.tight_layout()
plt.show()

print('\n📊 Average sales by holiday flag:')
print(avg_sales.rename({0: 'Non-Holiday', 1: 'Holiday'}).to_string())

---

## 📊 Section 10 — EDA: MarkDown vs Weekly Sales (Before vs After Nov 2011)

### 🎯 Objective
Analyze the relationship between `MarkDown` (total promotions) and `Weekly_Sales`, split by a key temporal boundary: **November 2011**.

### 🧠 Why?
MarkDown data is only reliably populated from November 2011 onwards. Splitting the data at this boundary reveals:
- Whether promotions had a measurable effect on sales.
- Whether the pre-Nov 2011 period (mostly zero MarkDown) behaves differently.

### ⚙️ What the code does
Splits `full_df` into `before_df` / `after_df` and plots scatter plots for each.

In [ ]:
# Ensure datetime
full_df['Date'] = pd.to_datetime(full_df['Date'])

before_df = full_df[full_df['Date'] <  '2011-11-01'].copy()
after_df  = full_df[full_df['Date'] >= '2011-11-01'].copy()

print(f'Records before Nov 2011 : {len(before_df):,}')
print(f'Records from  Nov 2011+ : {len(after_df):,}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MarkDown vs Weekly Sales — Time Split', fontsize=14, fontweight='bold')

for ax, df, title in [
    (axes[0], before_df, 'Before Nov 2011 (MarkDown mostly zero)'),
    (axes[1], after_df,  'From Nov 2011 onwards (MarkDown active)')
]:
    sns.scatterplot(
        x='MarkDown', y='Weekly_Sales',
        data=df, alpha=0.3, ax=ax
    )
    ax.set_title(title)
    ax.set_xlabel('MarkDown (normalized)')
    ax.set_ylabel('Weekly Sales')

plt.tight_layout()
plt.show()

print(f'\nMax MarkDown value in full_df: {full_df["MarkDown"].max():.4f}')

---

## 📊 Section 11 — EDA: MarkDown vs Weekly Sales (Holiday vs Non-Holiday)

### 🎯 Objective
Examine whether promotions (`MarkDown`) have a stronger effect on sales during **holiday** vs **non-holiday** weeks.

### 🧠 Why?
Promotional strategies likely amplify during holiday seasons. Understanding this interaction is valuable for feature engineering and model interpretation.

### ⚙️ What the code does
Splits `full_df` by `IsHoliday` and creates two scatter plots side by side.

In [ ]:
holiday_df     = full_df[full_df['IsHoliday'] == 1].copy()
non_holiday_df = full_df[full_df['IsHoliday'] == 0].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MarkDown vs Weekly Sales — Holiday Split', fontsize=14, fontweight='bold')

for ax, df, title in [
    (axes[0], holiday_df,     '🎉 Holiday Weeks'),
    (axes[1], non_holiday_df, '📅 Non-Holiday Weeks')
]:
    sns.scatterplot(
        x='MarkDown', y='Weekly_Sales',
        data=df, alpha=0.3, ax=ax
    )
    ax.set_title(title)
    ax.set_xlabel('MarkDown (normalized)')
    ax.set_ylabel('Weekly Sales')

plt.tight_layout()
plt.show()

---

## 📊 Section 12 — EDA: Economic Features vs Weekly Sales (Holiday Split)

### 🎯 Objective
Analyze how external economic factors (`Temperature`, `Fuel_Price`, `CPI`, `Unemployment`) relate to `Weekly_Sales`, broken down by holiday flag.

### 🧠 Why?
- High fuel prices may suppress shopping trips.
- High unemployment may reduce discretionary spending.
- Temperature may reflect seasonal demand shifts.
- CPI tracks consumer purchasing power.

Splitting by `IsHoliday` reveals whether these effects are amplified or muted during holidays.

### ⚙️ What the code does
Generates an 8-subplot grid (4 features × 2 holiday groups), with scatter plots and axis labels.

In [ ]:
features_to_plot = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

fig, axes = plt.subplots(4, 2, figsize=(16, 20))
fig.suptitle('Economic Features vs Weekly Sales\n(Left: Holiday | Right: Non-Holiday)',
             fontsize=14, fontweight='bold')

for i, feature in enumerate(features_to_plot):
    # ── Holiday ───────────────────────────────────────────────────────────────
    sns.scatterplot(
        x=feature, y='Weekly_Sales',
        data=holiday_df, alpha=0.3,
        ax=axes[i, 0]
    )
    axes[i, 0].set_title(f'{feature} vs Sales (Holiday)')
    axes[i, 0].set_xlabel(f'{feature} (normalized)')
    axes[i, 0].set_ylabel('Weekly Sales')

    # ── Non-Holiday ───────────────────────────────────────────────────────────
    sns.scatterplot(
        x=feature, y='Weekly_Sales',
        data=non_holiday_df, alpha=0.3,
        ax=axes[i, 1]
    )
    axes[i, 1].set_title(f'{feature} vs Sales (Non-Holiday)')
    axes[i, 1].set_xlabel(f'{feature} (normalized)')
    axes[i, 1].set_ylabel('Weekly Sales')

plt.tight_layout()
plt.show()

---

## ✅ Section 13 — Final Dataset Summary

### 🎯 Objective
Confirm the final `full_df` is correct, clean, and ready for ML model training.

### ⚙️ What the code does
Prints shape, dtypes, a statistical summary, and verifies all expected columns are present.

In [ ]:
expected_cols = ['Weekly_Sales', 'MarkDown', 'Temperature', 'Fuel_Price',
                 'CPI', 'Unemployment', 'IsHoliday']

print('=' * 55)
print('        FINAL DATASET SUMMARY (full_df)')
print('=' * 55)
print(f'Shape         : {full_df.shape}')
print(f'Columns       : {full_df.columns.tolist()}')
print(f'Total NaN     : {full_df.isnull().sum().sum()}')
print()

print('── Expected Columns Check ──────────────────────')
for col in expected_cols:
    status = '✅' if col in full_df.columns else '❌ MISSING'
    print(f'  {status}  {col}')

print()
print('── Dtypes ──────────────────────────────────────')
print(full_df.dtypes)

In [ ]:
full_df.describe().round(4)

In [ ]:
full_df.head(10)